# MCD-rPPG → 可训练数据集 构建器（Colab 流式版）

**目标**：把 HF 上 `kyegorov/mcd_rppg` 的 **135 GB 原始视频**，流式地"下载一段 → 解帧 → 68 点关键点 → 裁脸/算 8 个 ROI 特征 → 编码/存 npz → 上传到你自己的 HF 数据集 → 删除本地"，
最终得到一个**只提取一次**的、可以直接拿去训练的库。

**产物结构（你的新 HF 数据集）**
```
roi/{pid}_{step}_{view}.npz     landmarks(T,68,2) ppg(T,) roi_rgb(T,8,3) fps ts ...   ~0.6 MB/段
faces/{pid}_{step}_{view}.mp4   裁剪+蒙版的 128x128 人脸视频 (x264 crf12 yuv444)      ~5-12 MB/段
frames/{...}_c00.npz            可选：不编码时的裸帧分块
index.parquet                   13 项生物指标 + QC(心率/SNR/漏检率) + fold + 状态
```
**特点**
- **流式**：磁盘占用 ≈ 1 段视频（<100 MB），Colab 免费版也能跑
- **可断点续跑**：每次以 HF 上已存在的 `roi/*.npz` / `faces/*.mp4` 为准跳过
- **不会重复提取**：以后训练只读这个 repo
- 与官方 `rppglib` **完全兼容**（`video / landmarks / ppg` 三件套同名同义）

**先做冒烟测试**：保持 `LIMIT = 5` 跑完，看最后打印的 `秒/段`，再决定 `LIMIT = 0`（全部）。

## 0. 环境准备
只补 `face-alignment`（作者用的就是它，68 点），其余用 Colab 自带，避免 `requirements.txt` 里 torch/numpy 版本打架。

In [ ]:
%%bash
pip -q install "face-alignment==1.4.1" "huggingface_hub>=0.27" hf_transfer
python - <<'EOF'
import face_alignment, huggingface_hub, cv2, numpy, pandas, scipy
print("face_alignment", face_alignment.__version__)
print("hub", huggingface_hub.__version__)
print("cv2", cv2.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
EOF

## 1. 配置（只有这一格需要改）

| 变量 | 说明 |
|---|---|
| `DST_REPO` | 你的目标数据集名，不存在会自动创建 |
| `VIEWS` | `["front"]` 先跑正脸（1200 段 / ~90 GB 原始）；跑通再加 `"left","right"`（共 3600 段 / 130 GB） |
| `SAVE_FRAMES` | `"video"` 编成 mp4（推荐，全时长 30 GB）；`"npz"` 裸帧分块（大，仅小规模）；`"off"` 只留 ROI 特征 |
| `MAX_FRAMES` | `None` = 全时长 3 min（数据量 ×8，推荐）；`600` = 复刻官方 20 s 风格 |
| `LIMIT` | `5` → 冒烟；`0` → 全部 |

⚠️ `yuv444p` 不要改成 `yuv420p`：rPPG 靠的就是色度上的微小变化，420 会把色度砍一半。

In [ ]:
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")   # 更快的上传，必须在 import hub 之前

# ================= 只改这里 =================
DST_REPO    = "你的HF用户名/mcd-rppg-cropped"
PRIVATE     = True
VIEWS       = ["front"]
STEPS       = ["before", "after"]
SAVE_FRAMES = "video"     # "video" | "npz" | "off"
SAVE_ROI    = True        # 32 维/帧 特征（训 SCNN/POS 只需要它）
FRAME_SIZE  = 128         # 裁剪后人脸边长
CRF         = 12          # x264 质量，0=无损
MAX_FRAMES  = None        # None=全时长；600=官方 20s
DECODE      = "disk"      # "disk"=下到磁盘再删(推荐) | "url"=ffmpeg 直读 HF(零落盘，仅 SAVE_FRAMES="off" 时生效)
UPLOAD_MEM  = True        # roi/*.npz 直接从内存传 HF，不落盘
LIMIT       = 5           # ★ 先 5 段测吞吐，再改 0
SHARD_SIZE  = 20          # 每 N 段提交一次 commit
BATCH       = 32          # face_alignment 批大小
LM_DEVICE   = "cuda"      # 无 GPU 改 "cpu"
COMPRESS    = True
WORK_DIR    = "/content/mcd_work"
SRC_REPO    = "kyegorov/mcd_rppg"
# ===========================================

for d in ["", "/src", "/out", "/idx"]:
    os.makedirs(WORK_DIR + d, exist_ok=True)
print("dst :", DST_REPO)
print("views:", VIEWS, "| frames:", SAVE_FRAMES, "| size:", FRAME_SIZE, "| limit:", LIMIT)
print("decode:", DECODE, "| upload in memory:", UPLOAD_MEM)

## 2. 登录 HF + 建 repo
Colab 左侧 🔑 Secrets 里加一个 `HF_TOKEN`（write 权限），或者跑的时候会弹 notebook_login。

In [ ]:
from huggingface_hub import HfApi, CommitOperationAdd, hf_hub_download, login
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    from huggingface_hub import notebook_login
    notebook_login()

login(token=HF_TOKEN)
api = HfApi()
api.create_repo(repo_id=DST_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True)
print("repo ready →", f"https://huggingface.co/datasets/{DST_REPO}")

## 3. 建任务清单（`db.csv` 是唯一的总表）
`db.csv` 里每行 = 一个 `pid × step × camera`，自带 13 项生物指标和 6 个文件路径。
fold 优先复用官方 `mcd_rppg.csv`（按病人 5 折），拿不到就按 `patient_id % 5` 自切——**保证同一人的 before/after 不会跨折泄漏**。

In [ ]:
import io, json, math, re, shutil, subprocess, sys, time
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import scipy.signal

SRC = SRC_REPO
db = pd.read_csv(hf_hub_download(SRC, "db.csv", repo_type="dataset"))
db["key"] = [f"{int(p):04d}_{s}_{v}" for p, s, v in zip(db.patient_id, db.step, db.view)]
db = db[db["view"].isin(VIEWS) & db["step"].isin(STEPS)].reset_index(drop=True)

fold_map = {}
try:
    off = pd.read_csv("https://raw.githubusercontent.com/ksyegorov/mcd_rppg/main/mcd_rppg.csv")
    off["pid"] = off["file"].str.extract(r"mcd_rppg__(\d+)__")[0].astype(int)
    off["stp"] = off["file"].str.extract(r"mcd_rppg__\d+__(\w+)\.npz")[0]
    fold_map = {(int(a), b): int(c) for a, b, c in zip(off.pid, off.stp, off.fold)}
    print("复用官方 fold:", len(fold_map), "条")
except Exception as e:
    print("官方 fold 不可用，改为 patient_id % 5 :", e)

db["fold"] = [fold_map.get((int(p), s), int(p) % 5) for p, s in zip(db.patient_id, db.step)]

BIOMARKERS = ["upper_ap", "lower_ap", "saturation", "temperature", "hemoglobin",
              "glycated_hemoglobin", "cholesterol", "respiratory", "rigidity",
              "pulse", "stress", "age", "bmi", "sex", "weight", "height"]
print(db.groupby("view").size().to_dict(), "| 待处理:", len(db))
print(db[["key", "camera", "fold", "pulse"]].head(4).to_string(index=False))

In [ ]:
build_budget()

## 4. 工具函数
关键点用 `face_alignment`(FAN, 68 点) —— 和作者同一套，所以仓库里 `SCNN_8rois.py` 的 8 个 ROI 索引可以**原样照搬**。
`POS` 按 `rppglib/models/torchPOS.py` 的语义实现（1.6 s 滑窗 + overlap-add + 0.75–3 Hz 带通），保证特征与官方一致。

In [ ]:
import torch
import face_alignment


def to_tensor(frames):
    # face_alignment 的批量接口要 (B,3,H,W) 的 RGB float tensor，不是 list[np.ndarray]
    return torch.stack([torch.from_numpy(np.ascontiguousarray(f)).permute(2, 0, 1) for f in frames])


_FA = None
def get_fa():
    global _FA
    if _FA is None:
        LT = getattr(face_alignment.LandmarksType, "TWO_D", None) or face_alignment.LandmarksType._2D
        dev = LM_DEVICE
        try:
            _FA = face_alignment.FaceAlignment(LT, device=dev, flip_input=False)
        except Exception as e:
            if str(dev) == "cpu":
                raise
            print(f"⚠️ face_alignment({dev}) 初始化失败：{type(e).__name__}: {e}")
            print("   自动退回 CPU（能跑但慢 10~30 倍；想用 GPU 见『常见坑』第 9 条）")
            globals()["LM_DEVICE"] = "cpu"
            _FA = face_alignment.FaceAlignment(LT, device="cpu", flip_input=False)
        print("face_alignment ready on", globals().get("LM_DEVICE", dev))
    return _FA

# === 与 rppglib/models/SCNN_8rois.py 完全一致（68 点索引）===
FACE              = [1,2,3,4,5,6,8,9,10,11,12,13,14,15,16,17,27,26,25,24,23,22,21,20,19,18]
NOSE              = [22,23,43,36,35,34,33,32,40]
LEFT_UPPER_CHEEK  = [1,37,42,41,40,32,3,2]
RIGHT_UPPER_CHEEK = [17,46,47,48,43,36,15,16]
LEFT_LOWER_CHEEK  = [3,32,51,50,49,5,4]
RIGHT_LOWER_CHEEK = [15,36,53,54,55,13,14]
LIPS              = [49,50,51,52,53,54,55,56,57,58,59,60]
CHIN              = [5,6,7,8,9,10,11,12,13,56,57,58,59,60]
REGIONS = [FACE, NOSE, LEFT_UPPER_CHEEK, RIGHT_UPPER_CHEEK,
           LEFT_LOWER_CHEEK, RIGHT_LOWER_CHEEK, LIPS, CHIN]


def iter_frame_batches(path, start, end, bs):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError("cannot open " + str(path))
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    i, buf = start, []
    while end is None or i < end:
        ok, fr = cap.read()
        if not ok:
            break
        buf.append(fr[..., ::-1].copy())      # BGR -> RGB
        i += 1
        if len(buf) == bs:
            yield buf
            buf = []
    if buf:
        yield buf
    cap.release()


def region_rgbs(img, lms):
    H, W = img.shape[:2]
    out = np.zeros((8, 3), np.float32)
    for k, seg in enumerate(REGIONS):
        mask = np.zeros((H, W), np.uint8)
        cv2.fillPoly(mask, [np.round(np.asarray(lms)[seg]).astype(np.int32)], 255)
        px = img[mask > 125]
        if px.size:
            out[k] = px.mean(0)
    return out


def mask_convex(frame, pts):
    m = np.zeros(frame.shape[:2], np.uint8)
    cv2.fillPoly(m, [cv2.convexHull(np.round(pts).astype(np.int32))], 255)
    frame *= (m > 125)[..., None]
    return frame


def pos_signal(rgb, fs):
    Cn = rgb / (rgb.mean(0, keepdims=True) + 1e-6)
    S = Cn @ np.array([[0, 1, -1], [-2, 1, 1], [-1, 2, -1]], np.float64).T
    n = len(Cn)
    L = min(max(int(round(1.6 * fs)), 8), n)
    win = np.lib.stride_tricks.sliding_window_view(S, L, axis=0)   # (n-L+1, 3, L)
    s0, s1 = win[:, 0, :], win[:, 1, :]
    alpha = s0.std(1) / (s1.std(1) + 1e-4)
    h = s0 + alpha[:, None] * s1
    h = h - h.mean(1, keepdims=True)
    H = np.zeros(n)
    m = n - L + 1
    for k in range(L):
        H[k:k + m] += h[:, k]
    if n > 16:
        sos = scipy.signal.butter(1, [0.75 / (fs / 2), 3.0 / (fs / 2)], btype="bandpass", output="sos")
        H = scipy.signal.sosfiltfilt(sos, H)
    return H.astype(np.float32)


def bandpass(x, fs, lo=0.5, hi=8.0, order=4):
    sos = scipy.signal.butter(order, [lo / (fs / 2), min(hi, fs / 2 * 0.99) / (fs / 2)],
                              btype="bandpass", output="sos")
    return scipy.signal.sosfiltfilt(sos, np.asarray(x, np.float64)).astype(np.float32)


def hr_bpm(x, fs, lo=0.5, hi=3.5):
    x = np.asarray(x, np.float64)
    x = x - x.mean()
    f, p = scipy.signal.periodogram(x, fs=fs, nfft=2048, detrend=False)
    m = (f >= lo) & (f <= hi)
    if not m.any():
        return 0.0, 0.0
    hr = f[m][p[m].argmax()] * 60
    snr = 10 * np.log10(p[m].max() / (p[m].mean() + 1e-12))
    return float(hr), float(snr)


def parse_meta(path, T, fps):
    try:
        ts = []
        with open(path) as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 2:
                    ts.append(dt.datetime.fromisoformat(parts[1]).timestamp())
        ts = np.asarray(ts, np.float64)
        if len(ts) < T:
            ts = np.pad(ts, (0, T - len(ts)), mode="edge")
        ts = ts[:T]
        return (ts - ts[0]).astype(np.float64), True
    except Exception:
        return np.arange(T, dtype=np.float64) / fps, False


def encode_cmd(out_path, size, fps):
    return ["ffmpeg", "-y", "-loglevel", "error",
            "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{size}x{size}",
            "-r", f"{fps:.6f}", "-i", "-",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", str(CRF),
            "-pix_fmt", "yuv444p", out_path]

# ============ 零落盘解码：ffmpeg 直读 HF 的 resolve URL ============
def probe_url(url):
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                          "-show_entries", "stream=width,height,r_frame_rate",
                          "-of", "csv=p=0", url],
                         capture_output=True, text=True, timeout=120).stdout.strip()
    w, h, rate = out.split(",")[:3]
    num, den = (rate.split("/") + ["1"])[:2]
    return int(w), int(h), (float(num) / float(den) if float(den) else 30.0)


def iter_url_batches(url, W, H, bs):
    # 直接把远程 avi 解成 rgb24 管道塞进内存，磁盘占用为 0
    p = subprocess.Popen(["ffmpeg", "-loglevel", "error", "-i", url,
                          "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
                         stdout=subprocess.PIPE, bufsize=10 ** 8)
    fs = W * H * 3
    buf = []
    while True:
        raw = p.stdout.read(fs)
        if len(raw) < fs:
            break
        buf.append(np.frombuffer(raw, np.uint8).reshape(H, W, 3).copy())
        if len(buf) == bs:
            yield buf
            buf = []
    if buf:
        yield buf
    p.stdout.close()
    p.wait()


def open_reader(r, need_two_pass):
    """返回 (reader(start, end, bs), W, H, fps, 本地源文件路径 or None)"""
    if DECODE == "url" and not need_two_pass:
        url = f"https://huggingface.co/datasets/{SRC}/resolve/main/{r.video}"
        W, H, fps = probe_url(url)
        return (lambda s, e, bs: iter_url_batches(url, W, H, bs)), W, H, fps, None
    p = hf_hub_download(SRC, r.video, repo_type="dataset", local_dir=f"{WORK_DIR}/src")
    cap = cv2.VideoCapture(p)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    if not fps or fps <= 1:
        fps = 30.0
    return (lambda s, e, bs: iter_frame_batches(p, s, e, bs)), W, H, fps, p


def build_budget():
    mb = {"front": 78.0, "left": 29.0, "right": 16.0}
    n = len(db)
    dl = sum(mb.get(v, 40.0) for v in db.view) / 1000.0
    passes = 2 if (SAVE_FRAMES != "off" and DECODE == "disk") else (2 if SAVE_FRAMES != "off" else 1)
    roi_gb = n * 0.6 / 1000.0
    if SAVE_FRAMES == "video":
        per = 12.0 if MAX_FRAMES is None else 4.0
    elif SAVE_FRAMES == "npz":
        per = (MAX_FRAMES or 4800) / 600.0 * 17.0
    else:
        per = 0.0
    frames_gb = n * per / 1000.0
    print(f"记录数            : {n}")
    print(f"需下载(一次性)    : {dl:6.1f} GB   {'(url 模式: 单遍)' if passes == 1 else '(两遍解码，磁盘只留 1 段)'}")
    print(f"磁盘峰值          : {0 if DECODE == 'url' else max(mb.get(v, 40.0) for v in db.view):>6.0f} MB")
    print(f"上传 HF — roi     : {roi_gb:6.2f} GB")
    if frames_gb:
        print(f"上传 HF — frames  : {frames_gb:6.2f} GB")
    print(f"上传 HF — 合计    : {roi_gb + frames_gb:6.2f} GB")


## 5. 单段处理：解码 → 关键点 → 裁剪/特征 → 编码

两遍解码（第一遍只取关键点和 ROI 均值，第二遍才写像素），**内存恒定**，与 3 分钟还是 3 小时无关。
ROI 均值在原始分辨率上算——裁剪只是平移，像素值不变，所以和官方"裁完再算"等价。

In [ ]:
def process_record(r):
    t0 = time.time()
    key = r.key
    info = dict(key=key, patient_id=int(r.patient_id), step=r.step, view=r.view,
                camera=r.camera, fold=int(r.fold), status="ok", err="")
    for b in BIOMARKERS:
        info[b] = getattr(r, b, None)

    src = f"{WORK_DIR}/src"
    src_p = hf_hub_download(SRC, r.ppg_sync, repo_type="dataset", local_dir=src)
    ppg_all = np.loadtxt(src_p, ndmin=2)[:, 0].astype(np.float32)   # ★ 第 0 列（npy 版极性是反的）
    os.unlink(src_p)
    n_max = len(ppg_all) if MAX_FRAMES is None else min(MAX_FRAMES, len(ppg_all))

    need_two_pass = SAVE_FRAMES != "off"
    reader, W, H, fps, local_src = open_reader(r, need_two_pass)
    info["decode"] = DECODE if local_src is None else "disk"

    # ---------- pass 1: 68 点 + 8 ROI 均值（ROI 不需要裁剪，裁只是平移）----------
    fa = get_fa()
    lms_l, roi_l, prev, miss = [], [], np.zeros((68, 2), np.float32), 0
    for batch in reader(0, n_max, BATCH):
        preds = fa.get_landmarks_from_batch(to_tensor(batch))
        for img, p in zip(batch, preds):
            if p is None or len(p) == 0:            # 批量接口找不到脸时返回 []，不是 None
                miss += 1
                p = prev
            else:
                p = np.asarray(p, np.float32)
                prev = p
            lms_l.append(p.copy())
            if SAVE_ROI:
                roi_l.append(region_rgbs(img, p))
    T = len(lms_l)
    if T < 30:
        raise RuntimeError(f"only {T} frames decoded")
    lms = np.asarray(lms_l, np.float32)
    ppg = ppg_all[:T]
    roi = np.asarray(roi_l, np.float32) if SAVE_ROI else None

    lm_i = lms.round().astype(np.int16)
    x0, x1 = max(0, int(lm_i[..., 0].min())), min(W, int(lm_i[..., 0].max()))
    y0, y1 = max(0, int(lm_i[..., 1].min())), min(H, int(lm_i[..., 1].max()))
    if x1 - x0 < 16 or y1 - y0 < 16:
        raise RuntimeError("bad bbox")
    hr_t, snr_t = hr_bpm(bandpass(ppg, fps), fps)
    info.update(T=T, Hh=y1 - y0, Ww=x1 - x0, face_square=int((y1 - y0) * (x1 - x0)),
                fps=round(float(fps), 4), miss=int(miss),
                hr_ppg=round(hr_t, 2), snr=round(snr_t, 2), hr_tone=getattr(r, "pulse", None))

    outs, local = [], []

    # ---------- 小文件：landmarks + ppg + roi + 时间戳（可直接内存上传）----------
    if SAVE_ROI:
        src_m = hf_hub_download(SRC, r.meta, repo_type="dataset", local_dir=src)
        ts, ts_ok = parse_meta(src_m, T, fps)
        os.unlink(src_m)
        payload = dict(
            landmarks=lm_i, ppg=ppg, roi_rgb=roi,
            roi_pos=np.stack([pos_signal(roi[:, k, :], fps) for k in range(8)], 1),
            fps=np.float32(fps), ts=ts, face_square=np.int32(info["face_square"]),
            n_missing=np.int32(miss), patient_id=np.int32(r.patient_id),
            step=np.array(r.step), view=np.array(r.view), camera=np.array(r.camera),
            hr_ppg=np.float32(hr_t), ts_ok=np.bool_(ts_ok),
        )
        if UPLOAD_MEM:
            bio = io.BytesIO()
            np.savez_compressed(bio, **payload)
            bio.seek(0)
            outs.append((bio, f"roi/{key}.npz"))
        else:
            f_roi = f"{WORK_DIR}/out/roi_{key}.npz"
            np.savez_compressed(f_roi, **payload)
            outs.append((f_roi, f"roi/{key}.npz"))
            local.append(f_roi)

    # ---------- pass 2: 裁剪 + 凸包蒙版 (+ 编码) ----------
    if SAVE_FRAMES != "off":
        S = int(FRAME_SIZE)
        proc, mp4 = None, None
        if SAVE_FRAMES == "video":
            mp4 = f"{WORK_DIR}/out/faces_{key}.mp4"
            proc = subprocess.Popen(encode_cmd(mp4, S, fps), stdout=subprocess.DEVNULL,
                                    stderr=subprocess.DEVNULL, stdin=subprocess.PIPE)
        buf, n, ci = [], 0, 0
        for batch in reader(0, T, 32):
            for img in batch:
                c = img[y0:y1, x0:x1].copy()
                c = mask_convex(c, lms[n] - np.array([x0, y0], np.float32))
                c = cv2.resize(c, (S, S), interpolation=cv2.INTER_AREA)
                if proc is not None:
                    proc.stdin.write(np.ascontiguousarray(c).tobytes())
                else:
                    buf.append(c)
                    if len(buf) >= 600:
                        f = f"{WORK_DIR}/out/frames_{key}_c{ci:02d}.npz"
                        (np.savez_compressed if COMPRESS else np.savez)(
                            f, video=np.stack(buf), fps=np.float32(fps))
                        outs.append((f, f"frames/{key}_c{ci:02d}.npz"))
                        local.append(f)
                        buf, ci = [], ci + 1
                n += 1
        if proc is not None:
            proc.stdin.close()
            proc.wait()
            outs.append((mp4, f"faces/{key}.mp4"))
            local.append(mp4)
        elif buf:
            f = f"{WORK_DIR}/out/frames_{key}_c{ci:02d}.npz"
            (np.savez_compressed if COMPRESS else np.savez)(
                f, video=np.stack(buf), fps=np.float32(fps))
            outs.append((f, f"frames/{key}_c{ci:02d}.npz"))
            local.append(f)
        if n != T:
            info["err"] = f"pass2 frames {n} != {T}"

    if local_src:
        local.append(local_src)

    info["sec"] = round(time.time() - t0, 1)
    return outs, info, local


## 6. 和官方 npz 逐位对账（强烈建议先跑这一格）
官方 `mcd_rppg.csv` 里每行的 `video_shape / face_square / ppg_*` 实际上是**前 600 帧**的统计。
如果这一格打印的两行数字能对上，说明你的预处理和作者**一模一样**，后面的产物可以放心用。

In [ ]:
def verify_repo_compat(pid=1020, step="after"):
    off = pd.read_csv("https://raw.githubusercontent.com/ksyegorov/mcd_rppg/main/mcd_rppg.csv")
    row = off[off.file.str.contains(f"mcd_rppg__{pid}__{step}\\.npz")].iloc[0]
    r = db[(db.patient_id == pid) & (db.step == step) & (db.view == "front")].iloc[0]
    src = f"{WORK_DIR}/src"
    v = hf_hub_download(SRC, r.video, repo_type="dataset", local_dir=src)
    p = hf_hub_download(SRC, r.ppg_sync, repo_type="dataset", local_dir=src)
    ppg = np.loadtxt(p, ndmin=2)[:600, 0].astype(np.float32)

    fa = get_fa()
    lms_l = []
    for batch in iter_frame_batches(v, 0, 600, BATCH):
        for img, pr in zip(batch, fa.get_landmarks_from_batch(to_tensor(batch))):
            lms_l.append(np.zeros((68, 2), np.float32) if (pr is None or len(pr) == 0) else np.asarray(pr, np.float32))
    lms = np.asarray(lms_l).round().astype(np.int16)
    x0, x1 = int(lms[..., 0].min()), int(lms[..., 0].max())
    y0, y1 = int(lms[..., 1].min()), int(lms[..., 1].max())
    print(f"ours : shape=({len(lms)}, {y1-y0}, {x1-x0}, 3)  face_square={(y1-y0)*(x1-x0)}  "
          f"ppg min/max/mean/std={ppg.min():.3f}/{ppg.max():.3f}/{ppg.mean():.3f}/{ppg.std():.3f}")
    print(f"repo : shape={row.video_shape}  face_square={row.face_square}  "
          f"ppg min/max/mean/std={row.ppg_min}/{row.ppg_max}/{row.ppg_mean:.3f}/{row.ppg_std:.3f}")

verify_repo_compat(1020, "after")

## 7. 主循环：分片处理 → 一次 commit → 删本地（可断点续跑）
- 每 `SHARD_SIZE` 段打包成 **一个 `create_commit`**（不是每文件一次，省配额也快）
- 上传成功才删本地；失败就停下，下次跑会自动跳过已上传的
- 会打印 `秒/段`，据此估总时长

In [ ]:
def du(path):
    tot = 0
    for root, _, fs in os.walk(path):
        for f in fs:
            try: tot += os.path.getsize(os.path.join(root, f))
            except Exception: pass
    return tot / 1e6


def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield i // n, seq[i:i + n]

done = set(api.list_repo_files(DST_REPO, repo_type="dataset"))
def is_done(k):
    return (f"roi/{k}.npz" in done) or (f"faces/{k}.mp4" in done) or (f"frames/{k}_c00.npz" in done)

todo = db[~db["key"].map(is_done)].reset_index(drop=True)
if LIMIT:
    todo = todo.head(LIMIT)
print(f"已完成 {len(db)-len(todo if not LIMIT else db[~db['key'].map(is_done)])} 段 | 本次待处理 {len(todo)} 段")

t_start, n_ok = time.time(), 0
for si, chunk in chunked(todo, SHARD_SIZE):
    ops, rows, to_del = [], [], []
    for r in chunk.itertuples():
        try:
            outs, info, local = process_record(r)
            ops += [CommitOperationAdd(path_in_repo=rp, path_or_fileobj=lp) for lp, rp in outs]
            to_del += local
            rows.append(info)
            n_ok += 1
            print(f'  ✓ {info["key"]}  T={info.get("T")} fps={info.get("fps")} '
                  f'HR={info.get("hr_ppg")}/{info.get("hr_tone")} miss={info.get("miss")} '
                  f'{info["sec"]}s{"  !"+info["err"] if info["err"] else ""}')
        except Exception as e:
            rows.append(dict(key=r.key, patient_id=int(r.patient_id), step=r.step,
                             view=r.view, status="error", err=str(e)[:200]))
            print("  ✗", r.key, type(e).__name__, str(e)[:120])

    pq = f"{WORK_DIR}/out/shard_{si:05d}.parquet"
    pd.DataFrame(rows).to_parquet(pq)
    ops.append(CommitOperationAdd(path_in_repo=f"index/shard_{si:05d}.parquet", path_or_fileobj=pq))

    for attempt in range(4):
        try:
            api.create_commit(repo_id=DST_REPO, repo_type="dataset", operations=ops,
                              commit_message=f"shard {si:05d}: {len(rows)} records")
            break
        except Exception as e:
            print("  上传重试", attempt, str(e)[:140])
            time.sleep(10 * (attempt + 1))
    else:
        print("!! 连续上传失败，停止（本地文件已保留本 shard，下次续跑）")
        break

    for f in to_del + [pq]:
        if isinstance(f, str):
            try: os.unlink(f)
            except Exception: pass
    for f in Path(f"{WORK_DIR}/src").glob("*"):
        try: f.unlink()
        except Exception: pass
    print(f"     磁盘占用 {du(WORK_DIR):.1f} MB")

    el = time.time() - t_start
    print(f"[shard {si:05d}] 累计 {n_ok} 段 | {el/60:.1f} min | 均 {el/max(n_ok,1):.0f} s/段 "
          f"| 预计剩余 {(len(todo)-n_ok)*el/max(n_ok,1)/3600:.1f} h")

## 8. 汇总索引 + 生成数据集卡
把所有 `index/shard_*.parquet` 合并成 `index.parquet`（含 13 项生物指标、心率/SNR/漏检率、fold），再写一份 dataset card。
之后任何人（包括你）只要 `pd.read_parquet("hf://datasets/<repo>/index.parquet")` 就能拿到全部元数据，**不必下载任何一个 npz**。

In [ ]:
rows = []
for f in sorted(api.list_repo_files(DST_REPO, repo_type="dataset")):
    if f.startswith("index/shard_"):
        rows.append(pd.read_parquet(hf_hub_download(DST_REPO, f, repo_type="dataset",
                                                   local_dir=f"{WORK_DIR}/idx")))
index = pd.concat(rows, ignore_index=True).drop_duplicates("key") if rows else pd.DataFrame()
if len(index):
    index.to_parquet(f"{WORK_DIR}/index.parquet")
    api.upload_file(path_or_fileobj=f"{WORK_DIR}/index.parquet", path_in_repo="index.parquet",
                    repo_id=DST_REPO, repo_type="dataset", commit_message="index")
    print(index.status.value_counts().to_dict(), "| 总记录", len(index))
    print(index[["key", "T", "fps", "face_square", "hr_ppg", "pulse", "snr", "fold"]].head(5).to_string(index=False))
else:
    print("还没有 shard 索引")

card = f'''---
license: cc-by-4.0
task_categories: [other]
tags: [rPPG, remote-photoplethysmography, video, health-biomarkers]
pretty_name: {DST_REPO.split("/")[-1]}
---

# MCD-rPPG 裁剪/特征版（由 ksyegorov/mcd_rppg 流式二次加工）

上游数据集：[kyegorov/mcd_rppg](https://huggingface.co/datasets/kyegorov/mcd_rppg)（600 人 × 2 状态 × 3 机位，CC-BY-4.0）
本 repo 是把上游 135 GB 原始 avi **只做一次**人脸提取后的产物，避免每次训练重复解码/检测。

## 文件
- `roi/{{pid}}_{{step}}_{{view}}.npz`：`landmarks(T,68,2) int16`、`ppg(T,) float32`（来自上游 `ppg_sync` 第 0 列，100 Hz→逐帧对齐）、
  `roi_rgb(T,8,3)`（8 个 ROI 的 RGB 均值）、`roi_pos(T,8)`（每 ROI 的 POS 脉搏波）、`fps`、`ts`、`face_square`
- `faces/{{pid}}_{{step}}_{{view}}.mp4`：裁剪+凸包蒙版后的 {FRAME_SIZE}×{FRAME_SIZE} 人脸视频（x264 crf{CRF} yuv444p）
- `index.parquet`：13 项生物指标 + QC（FFT 心率、SNR、关键点漏检数）+ fold（按病人划分，无泄漏）

## 用法
```python
import pandas as pd
from huggingface_hub import hf_hub_download
idx = pd.read_parquet(hf_hub_download("{DST_REPO}", "index.parquet", repo_type="dataset"))
x = hf_hub_download("{DST_REPO}", "roi/1020_after_front.npz", repo_type="dataset")
```

## 引用
Egorov et al., *Gaze into the Heart: A Multi-View Video Dataset for rPPG and Health Biomarkers Estimation*, ACM MM 2025 (arXiv:2508.17924)
'''
api.upload_file(path_or_fileobj=io.BytesIO(card.encode()), path_in_repo="README.md",
                repo_id=DST_REPO, repo_type="dataset", commit_message="dataset card")
print("README.md 已上传")

## 9. 产物自检：直接当 torch Dataset 用
验证 32 维/帧特征能被官方 `SCNN_8roi_NN` 吃下去（输入 `(B, T, 32)`）。

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class RoiDataset(Dataset):
    def __init__(self, index, repo, window=300, train=True):
        self.index = index.reset_index(drop=True)
        self.repo, self.window, self.train = repo, window, train

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        r = self.index.iloc[i]
        f = hf_hub_download(self.repo, f"roi/{r.key}.npz", repo_type="dataset",
                            local_dir=f"{WORK_DIR}/cache")
        z = np.load(f)
        rgb, pos, ppg = z["roi_rgb"], z["roi_pos"], z["ppg"]
        x = np.concatenate([rgb.reshape(len(rgb), -1),
                            pos.reshape(len(pos), -1)], 1).astype(np.float32)   # (T, 32)
        T = min(len(x), len(ppg))
        if T > self.window:
            s = np.random.randint(T - self.window) if self.train else 0
            x, ppg = x[s:s + self.window], ppg[s:s + self.window]
        ppg = ppg - ppg.mean()
        ppg = ppg / (ppg.std() + 1e-9)
        return torch.from_numpy(x), torch.from_numpy(ppg.astype(np.float32))

if len(index):
    ds = RoiDataset(index[index.status == "ok"], DST_REPO)
    dl = DataLoader(ds, batch_size=2)
    xb, yb = next(iter(dl))
    print("x", tuple(xb.shape), "| y", tuple(yb.shape), "→ 送进 rppglib.models.SCNN_8roi_NN 即可")
else:
    print("先跑第 7 格")

## 10. 在 Colab 上跑官方训练代码的补丁（重要）

官方 `train_SCNN_8roi_mcd_rppg.ipynb` 直接跑会挂，因为路径写死在作者的服务器上。把下面这几行放在它前面：

```python
import os, sys, subprocess
subprocess.run(["git","clone","--depth","1","https://github.com/ksyegorov/rppg_dummy"], check=False)
# 1) 仓库代码
!git clone -q https://github.com/ksyegorov/mcd_rppg.git && cd mcd_rppg && pip -q install face-alignment timm
sys.path.insert(0, "/content/mcd_rppg"); os.chdir("/content/mcd_rppg")

# 2) 拿你自己产出的 npz（一次性快照，之后不再碰原始 135 GB）
from huggingface_hub import snapshot_download
LOCAL = "/content/ds"
snapshot_download("你的HF用户名/mcd-rppg-cropped", repo_type="dataset",
                  local_dir=LOCAL, allow_patterns=["roi/*.npz"])

# 3) rppglib/params.py 里的 /gim/lv03/... 全部替换
import rppglib.params as P
P.PREPROCESSED_PATH = LOCAL
P.DATASET_PATHS["mcd_rppg"] = LOCAL

# 4) 官方的 csv 里 file 列是绝对路径 → 改写成指向你的 roi npz（字段名对齐即可）
import pandas as pd
df = pd.read_csv("mcd_rppg.csv")
df["file"] = df["file"].str.replace("/gim/lv03/datasets/rppg/preprocessed/mcd_rppg__", LOCAL + "/roi/", regex=False)
df["file"] = [f.replace(".npz", "_front.npz") for f in df["file"]]
df.to_csv("mcd_rppg.csv", index=False)

# 5) SCNN 的 video_processing 要写 cache/ 目录，先建好，否则直接崩
os.makedirs("cache", exist_ok=True)
```

然后再跑 `train_SCNN_8roi_mcd_rppg.ipynb`。

**小技巧**：`cache/` 里缓存的就是那 32 维特征（`md5.npy`）。因为我们的 `roi_*.npz` 已经把 `roi_rgb/roi_pos` 算好了，
你也可以**绕开整个官方预处理**，直接用第 9 格的 `RoiDataset` + `rppglib.models.SCNN_8rois.SCNN_8roi_NN` 自己写 20 行训练循环——
不需要原图、不需要 face_alignment，**纯 CPU 就能训**（每段只有 600×32 个数）。

## 11. 扩容清单

| 想做什么 | 改什么 |
|---|---|
| 跑全部三机位 | `VIEWS = ["front","left","right"]`，`LIMIT = 0` |
| 全时长（数据 ×8） | `MAX_FRAMES = None`（默认已是） |
| 只要 ROI 特征，不要像素 | `SAVE_FRAMES = "off"` → 全库只有 ~2.2 GB，几小时跑完 |
| 复刻官方 20 s 版本 | `MAX_FRAMES = 600` + `SAVE_FRAMES = "npz"` + `FRAME_SIZE = None`(需改代码保留原生尺寸) |
| 想无损 | `CRF = 0`（体积 ×3~4） |
| 加 ECG | 见下一格 |

⚠️ 时间预估（实测值请以第 7 格打印的 `s/段` 为准）：正脸 1200 段约 **20~30 GPU 小时**，三机位 3600 段约 **60~90 GPU 小时**。
一定要先 `LIMIT = 5` 测一次。

## 12.（可选）ECG：3.8 GB JSON → 1.4 GB npy
500 Hz，`{frequency, dataX[], dataY[]}`，每人 2 个文件（与机位无关）。若要做 ECG 监督或多模态对齐再跑。

In [ ]:
def convert_ecg(limit=None):
    src = hf_hub_download(SRC, "db.csv", repo_type="dataset")
    d = pd.read_csv(src).drop_duplicates(subset=["ecg"])
    if limit:
        d = d.head(limit)
    for i, r in enumerate(d.itertuples()):
        f = hf_hub_download(SRC, r.ecg, repo_type="dataset", local_dir=f"{WORK_DIR}/src")
        j = json.load(open(f))
        y = np.asarray(j["dataY"], np.float32)
        name = Path(r.ecg).stem                      # {pid}_{step}
        out = f"{WORK_DIR}/out/ecg_{name}.npy"
        np.save(out, y)
        api.upload_file(path_or_fileobj=out, path_in_repo=f"ecg/{name}.npy",
                        repo_id=DST_REPO, repo_type="dataset", commit_message=f"ecg {name}")
        os.unlink(out); os.unlink(f)
        print(i, name, y.shape, f"{j['frequency']} Hz", f"{len(y)/j['frequency']:.1f} s")

# convert_ecg(limit=3)

---
## 备注：上游数据的几个坑（我在实测中确认的）

1. **`ppg_sync` 已经是逐帧对齐的**：`1020_FullHDwebcam_after.avi` 解码 5383 帧，`ppg_sync/1020_FullHDwebcam_after.txt` 正好 5383 行，第 0 列就是该帧的 PPG 值 —— **不需要自己做视频↔PPG 同步**。
2. **`ppg_sync_npy` 的极性是反的**：`col0 + npy ≡ 236`。官方 `mcd_rppg.csv` 的 ppg 统计与 **`ppg_sync` 第 0 列**吻合，所以本 notebook 只读 txt。
3. **三机位帧率不同**：FullHDwebcam 29.97 / USBVideo 30 / IriunWebcam 24。每个 npz 都存了 `fps`，训练时窗口长度按秒算，不要按帧数硬编码。
4. **官方 npz 只用了前 600 帧（20 s）**，3 分钟里扔掉了 140 秒 —— 你的 `MAX_FRAMES=None` 版本数据量直接 ×8。
5. 视频有坏帧（ffprobe 会报 `Error at MB`），解码帧数偶尔差几帧，代码里已用 `T = min(帧数, ppg 行数)` + 记录 `err` 兜住。
6. `db.csv` 同一病人 6 行（3 机位 × 2 状态），`weight/height/bmi/age/sex` 等是**重复值**，训练回归时别重复计数。